# Grounded Quiz Tutor — run on Google Colab

Generate quiz questions from lecture notes with an **open LLM**, check that every question is supported by the
lecture, and practise with an AI tutor that grades answers and asks follow-up questions.

**Before you start:** `Runtime → Change runtime type → T4 GPU → Save`. Then run the cells from top to bottom.

## 1. Get the code
If the project is on GitHub, keep `USE_GITHUB = True`. Otherwise set it to `False` and upload `grounded-quiz-tutor.zip` when asked.

In [ ]:
import os

USE_GITHUB = True
REPO = "https://github.com/quratulain77-saeed/grounded-quiz-tutor.git"

os.chdir("/content")
if not os.path.exists("/content/grounded-quiz-tutor"):
    if USE_GITHUB:
        !git clone -q {REPO}
    else:
        from google.colab import files
        files.upload()                      # choose grounded-quiz-tutor.zip
        !unzip -q -o grounded-quiz-tutor.zip
os.chdir("/content/grounded-quiz-tutor")
print("Working in:", os.getcwd())

## 2. Install dependencies
`torch` and `transformers` are already installed on Colab.

In [ ]:
!pip install -q gradio pypdf huggingface_hub accelerate pytest
!nvidia-smi -L || echo "No GPU found: use Runtime > Change runtime type > T4 GPU (the app still works, but slowly)"

## 3. Choose the model

* **Open model on the Colab GPU** (default, no account needed): `Qwen/Qwen2.5-3B-Instruct` fits on a free T4.
  Use `Qwen/Qwen2.5-1.5B-Instruct` for faster but simpler questions.
* **Hugging Face Inference API** (optional): add a secret named `HF_TOKEN` in the 🔑 *Secrets* panel on the left.

In [ ]:
os.environ["QT_LOCAL_MODEL"] = "Qwen/Qwen2.5-3B-Instruct"
os.environ["QT_DEFAULT_BACKEND"] = "Open model on this machine / Colab GPU"

try:  # optional: Hugging Face token from Colab secrets
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN loaded from Colab secrets")
except Exception:
    print("No HF_TOKEN secret (fine: the local Colab GPU model will be used)")

## 4. Run the tests (offline, about 10 seconds)

In [ ]:
!python -m pytest -q tests

## 5. Quick check without the web app
Loads the model (about 1–2 minutes the first time), generates a quiz from the sample lecture and grades one answer.

In [ ]:
from quiztutor.llm import make_llm
from quiztutor.generator import generate_quiz
from quiztutor.tutor import assess

llm = make_llm("local")
lecture = open("examples/sample_lecture.txt", encoding="utf-8").read()

questions = generate_quiz(llm, lecture, n=5, keep_ungrounded=True)
for i, q in enumerate(questions, 1):
    status = "grounded" if q.grounded else "NOT FOUND in lecture"
    print(f"{i}. [{q.type}, {q.difficulty}] {q.question}")
    if q.options:
        print("   options:", q.options)
    print(f"   answer: {q.answer}   |   evidence: {status} ({q.grounding_score:.2f})\n")

if questions:
    result = assess(llm, questions[0], "my test answer")
    print("Verdict:", result.verdict, "| Feedback:", result.feedback, "| Follow-up:", result.follow_up)

## 6. Launch the web app
Click the **public URL** (`https://….gradio.live`) that appears below. Stop the cell (■) to shut the app down.

In [ ]:
import app
app.demo.queue().launch(share=True, debug=True)

---
**Tips**
* Paste your own lecture notes or upload a `.txt` / `.pdf` in the *Lecture* tab.
* If generation is slow, switch to `Qwen/Qwen2.5-1.5B-Instruct` in step 3 and rerun steps 3, 5 and 6.
* If you see *CUDA out of memory*: `Runtime → Restart session`, choose the smaller model, and run again.